1. Load libraries

In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import minmax_scale
import matplotlib.pyplot as plt

2. Load Data

In [2]:
# Read in all .csv files from a folder and store them in a dictionary
def read_csv_files(folder_path):
    dataframes = {}
    for file_name in os.listdir(folder_path):
        if file_name.endswith(".csv"):
            file_path = os.path.join(folder_path, file_name)
            dataframe = pd.read_csv(file_path, header=None)  # First row is x-axis
            dataframes[file_name] = dataframe
    return dataframes

# Define the folder paths (must be folders, not files)
folder_path = "csv files"     

# Load the data
dataframe = read_csv_files(folder_path)


3. Filter Dataframes (400 rel/cm-1 to 2000 rel/cm-1)

In [3]:
def filter_columns_by_x_axis(dataframe, lower_bound, upper_bound):
    # Extract the x-axis (first row of the .csv file)
    x_axis = dataframe.iloc[0, :]
    # Create a mask for columns within the desired range
    mask = (x_axis >= lower_bound) & (x_axis <= upper_bound)
    # Apply the mask to filter columns
    filtered_df = dataframe.loc[:, mask]
    return filtered_df

# Define the folder paths
input_folder = "csv files" 
output_folder = "csv files/Fingerprint Region"

os.makedirs(output_folder, exist_ok=True)

# Load all CSV files
dataframes = read_csv_files(input_folder)

# Process each DataFrame
for file_name, df in dataframes.items():
    filtered_df = filter_columns_by_x_axis(df, 400, 2000)
    
    # Save the filtered DataFrame to the output folder
    output_file = os.path.join(output_folder, f"{file_name}")
    filtered_df.to_csv(output_file, header=False, index=False)  # Save without headers or index
    print(f"Filtered data saved to: {output_file}")

Filtered data saved to: csv files/Fingerprint Region\rep1_mixed_1-9_S-P_spectra.csv
Filtered data saved to: csv files/Fingerprint Region\rep1_mixed_3-7_S-P_cropped_spectra.csv
Filtered data saved to: csv files/Fingerprint Region\rep1_mixed_5-5_S-P_cropped_spectra.csv
Filtered data saved to: csv files/Fingerprint Region\rep1_mixed_7-3_S-P_spectra.csv
Filtered data saved to: csv files/Fingerprint Region\rep1_mixed_9-1_S-P_spectra.csv
Filtered data saved to: csv files/Fingerprint Region\rep1_single_PAO1_cropped_bottom_spectra.csv
Filtered data saved to: csv files/Fingerprint Region\rep1_single_PAO1_cropped_top_spectra.csv
Filtered data saved to: csv files/Fingerprint Region\rep1_single_STAPH_spectra.csv
Filtered data saved to: csv files/Fingerprint Region\rep2_mixed_1-9_S-P_cropped_bottom_spectra.csv
Filtered data saved to: csv files/Fingerprint Region\rep2_mixed_1-9_S-P_cropped_top_left_spectra.csv
Filtered data saved to: csv files/Fingerprint Region\rep2_mixed_1-9_S-P_cropped_top_right_

4. Outlier Removal followed by Min-Max Normalization

In [4]:
# Function for replacing outlier (using IQR method)
def replace_outliers(df):
    for column in df.columns:
        q1 = df[column].iloc[1:].quantile(0.25)
        q3 = df[column].iloc[1:].quantile(0.75)
        iqr = q3 - q1

        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        median = df[column].iloc[1:].median()

        df.loc[1:, column] = np.where(
            (df.loc[1:, column] < lower) | (df.loc[1:, column] > upper),
            median,
            df.loc[1:, column]
        )
    return df

#Function for Min-Max normalization
def min_max(df):
    # Exclude the first row (header) and normalize the rest of the data (starting from row 1)
    data_to_normalize = df.iloc[1:, :]  # Exclude the first row (header)
    # Check if there is any valid data to normalize
    if data_to_normalize.empty or data_to_normalize.isnull().all().all():
        print(f"No valid data to normalize in {df.columns[0]}...")
        return df  # Return the original dataframe if no valid data is present
    # Normalize across rows (axis=1 means normalize each row)
    normalized_data = minmax_scale(data_to_normalize, axis=1)  # Normalize each row
    # Convert the normalized data back to a DataFrame with the original columns
    df_normalized = pd.DataFrame(normalized_data, columns=df.columns)
    
    # Insert the first row (header) back to keep it as the header
    df_normalized.loc[-1] = df.iloc[0, :]  # Insert the first row (header) at the top
    df_normalized.index = df_normalized.index + 1  # Shift the index by 1
    df_normalized.sort_index(inplace=True)  # Sort the DataFrame by index
    df_normalized = df_normalized.drop(index=0)
    return df_normalized


# Define folder path for the original files
input_folder = "csv files/Fingerprint Region" 

# Create output folder for normalized and outlier-corrected files
output_folder = "csv files/Fingerprint Region/ Outlier Removed and Normalized" #- replace with your actual path
os.makedirs(output_folder, exist_ok=True)

# Iterate through all CSV files in the input folder
for filename in os.listdir(input_folder):
    if filename.endswith('.csv'):
        # Define the full path to the current CSV file
        file_path = os.path.join(input_folder, filename)
        # Read the CSV file into a DataFrame
        df = pd.read_csv(file_path)
        # Apply the outlier replacement
        df_outlier_corrected = replace_outliers(df.copy())
        # Apply Min-Max normalization
        df_normalized = min_max(df_outlier_corrected.copy())
        # Define the output file path with the name added at the end
        output_file_path = os.path.join(output_folder, f'{filename[:-4]}_normalized_outlier_corrected.csv')
        # Save the normalized and outlier-corrected DataFrame to a new CSV file
        df_normalized.to_csv(output_file_path, index=False)
        print(f'Processed file saved: {output_file_path}')

print('Processing complete for all CSV files.')

Processed file saved: csv files/Fingerprint Region/ Outlier Removed and Normalized\rep1_mixed_1-9_S-P_spectra_normalized_outlier_corrected.csv
Processed file saved: csv files/Fingerprint Region/ Outlier Removed and Normalized\rep1_mixed_3-7_S-P_cropped_spectra_normalized_outlier_corrected.csv
Processed file saved: csv files/Fingerprint Region/ Outlier Removed and Normalized\rep1_mixed_5-5_S-P_cropped_spectra_normalized_outlier_corrected.csv
Processed file saved: csv files/Fingerprint Region/ Outlier Removed and Normalized\rep1_mixed_7-3_S-P_spectra_normalized_outlier_corrected.csv
Processed file saved: csv files/Fingerprint Region/ Outlier Removed and Normalized\rep1_mixed_9-1_S-P_spectra_normalized_outlier_corrected.csv
Processed file saved: csv files/Fingerprint Region/ Outlier Removed and Normalized\rep1_single_PAO1_cropped_bottom_spectra_normalized_outlier_corrected.csv
Processed file saved: csv files/Fingerprint Region/ Outlier Removed and Normalized\rep1_single_PAO1_cropped_top_s